# 108 — Memoria de corto y largo plazo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

El LLM es **sin estado**: la memoria está en el sistema que lo rodea.

- **Corto plazo** = el hilo (*thread*): mensajes reenviados en cada llamada. Ventana
  finita, coste lineal en tokens, muere con el hilo.
- **Largo plazo** = el *store* externo, con dos operaciones: escribir (qué persiste) y
  leer (retrieval de las clases 100-104 aplicado a los propios recuerdos). Taxonomía:
  **episódica** (eventos fechados), **semántica** (hechos estables destilados),
  **procedimental** (reglas de comportamiento — la de mayor radio de daño si se corrompe).
- **Compactación**: al crecer el hilo, un LLM resume + extrae hechos → store; el hilo
  queda como `[resumen] + últimos n turnos`. Es **con pérdida**: lo omitido deja de
  existir para el sistema. El olvido (TTL, sobrescritura, decaimiento) es una función,
  no un fallo.

Referencias: Generative Agents (arXiv:2304.03442), MemGPT (arXiv:2310.08560),
LangGraph Memory docs.

## 🧮 Ejemplo de referencia

Hilo de soporte de 12 turnos ≈ 5 200 tokens, presupuesto 4 000:

```text
resumen (180 tok): "Ana, error 502 en 'pagos' tras v2.3.1; causa DB_POOL;
                    pendiente: verificar en staging."
semántica:  {usuario: Ana, rol: DevOps}, {servicio: pagos, versión: v2.3.1}
episódica:  "2026-07-30: 502 resuelto migrando DB_POOL"

hilo nuevo = resumen + últimos 4 turnos ≈ 1 080 tok   (compresión ~79 %)
```

Tres semanas después, "otra vez un 502 en pagos" recupera el episodio por similitud y
propone revisar DB_POOL. Si el resumen hubiera omitido el pendiente de staging, la
pregunta "¿me confirmas lo de staging?" habría fallado: la calidad de la compactación
**es** la calidad de la memoria.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("agent", seed=108)
show(result)


## Reflexión

1. ¿Por qué una ventana de contexto de un millón de tokens no elimina la necesidad de memoria de largo plazo? Da al menos dos razones independientes (coste, persistencia, atención).
2. En el ejemplo, ¿qué política aplicarías si la compactación extrae "prefiere respuestas detalladas" y el store ya contiene "prefiere brevedad"? Compara sobrescribir, versionar y preguntar.
3. ¿Por qué la memoria procedimental exige más control de escritura que la episódica? Razona en términos de radio de daño de un error.